# Question B4 (10 marks)

Model degradation is a common issue faced when deploying machine learning models (including neural networks) in the real world. New data points could exhibit a different pattern from older data points due to factors such as changes in government policy or market sentiments. For instance, housing prices in Singapore have been increasing and the Singapore government has introduced 3 rounds of cooling measures over the past years (16 December 2021, 30 September 2022, 27 April 2023).

In such situations, the distribution of the new data points could differ from the original data distribution which the models were trained on. Recall that machine learning models often work with the assumption that the test distribution should be similar to train distribution. When this assumption is violated, model performance will be adversely impacted.  In the last part of this assignment, we will investigate to what extent model degradation has occurred.




---



---



Your co-investigators used a linear regression model to rapidly test out several combinations of train/test splits and shared with you their findings in a brief report attached in Appendix A below. You wish to investigate whether your deep learning model corroborates with their findings.

In [3]:
# !pip install alibi-detect

In [4]:
# !pip install pytorch_tabular[extra]

In [5]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

from alibi_detect.cd import TabularDrift

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder

## Configuration for Google Colab
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/SC4001 Neural Network')

Mounted at /content/drive


1.Evaluate your model from B1 on data from year 2022 and report the test R2.

In [6]:
df = pd.read_csv('/content/drive/MyDrive/SC4001 Neural Network/hdb_price_prediction.csv')

# TODO: Enter your code here

# Load the model trained from B1
b1_tabular_model = TabularModel.load_model('/content/drive/MyDrive/SC4001 Neural Network/b1model')

# Define features that should be used
numeric_features     = ['dist_to_nearest_stn', 'dist_to_dhoby', 'degree_centrality', 'eigenvector_centrality', 'remaining_lease_years', 'floor_area_sqm']
categorical_features = ["month", "town", "flat_model_type", "storey_range"]
prediction_target    = ['resale_price']
all_features         = numeric_features + categorical_features + prediction_target

# Split features for train, validation & test by year
train_df      = df[df['year'] <= 2019]
val_df        = df[df['year'] == 2020]
test_df_2021  = df[df['year'] == 2021]
test_df_2022  = df[df['year'] == 2022]
test_df_2023  = df[df['year'] == 2023]

train_df      = train_df[all_features]
val_df        = val_df[all_features]
test_df_2021  = test_df_2021[all_features]
test_df_2022  = test_df_2022[all_features]
test_df_2023  = test_df_2023[all_features]


/usr/lib/python3.12/pickle.py:1557: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  obj = cls.__new__(cls, *args)
2026-03-14 13:16:58,172 - {pytorch_tabular.tabular_model:170} - INFO - Experiment Tracking is turned off
2026-03-14 13:16:58,177 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and ar

In [7]:
evaluation_2022 = b1_tabular_model.evaluate(test_df_2022)
pred_df_2022 = b1_tabular_model.predict(test_df_2022)

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │       17054725120.0       │
│  test_mean_squared_error  │       17054725120.0       │
└───────────────────────────┴───────────────────────────┘

In [8]:
test_values_2022  = test_df_2022["resale_price"].values
pred_values_2022  = pred_df_2022["resale_price_prediction"].values

# Calculate MSE and R2
test_mse_2022 = mean_squared_error(test_values_2022, pred_values_2022)
test_r2_2022  = r2_score(test_values_2022, pred_values_2022)

print(f"2023 Test MSE: {test_mse_2022:.4f}")
print(f"2023 Test R2:  {test_r2_2022:.4f}")

2023 Test MSE: 17054725204.2593
2023 Test R2:  0.4117


2.Evaluate your model from B1 on data from year 2023 and report the test R2.

In [9]:
# TODO: Enter your code here
evaluation_2023 = b1_tabular_model.evaluate(test_df_2023)
pred_df_2023 = b1_tabular_model.predict(test_df_2023)

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │       25678397440.0       │
│  test_mean_squared_error  │       25678397440.0       │
└───────────────────────────┴───────────────────────────┘

In [10]:
test_values_2023  = test_df_2023["resale_price"].values
pred_values_2023  = pred_df_2023["resale_price_prediction"].values

# Calculate MSE and R2
test_mse_2023 = mean_squared_error(test_values_2023, pred_values_2023)
test_r2_2023  = r2_score(test_values_2023, pred_values_2023)

print(f"2023 Test MSE: {test_mse_2023:.4f}")
print(f"2023 Test R2:  {test_r2_2023:.4f}")

2023 Test MSE: 25678395498.5468
2023 Test R2:  0.1290


3.Did model degradation occur for the deep learning model?


In [11]:
# YOUR ANSWER HERE
results_df = pd.DataFrame({
    'Test Year': [2021, 2022, 2023],
    'MSE': [6372397896.7095, test_mse_2022, test_mse_2023],
    'R2':  [0.7591, test_r2_2022, test_r2_2023]
})

results_df.index      = results_df['Test Year']
results_df            = results_df.drop(columns = ['Test Year'])
results_df.index.name = 'Test Year'

display(results_df.style.format({
    'MSE': '{:,.4f}',
    'R2':  '{:.4f}'
}).set_caption('Model Performance Across Test Years').set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('padding', '10px 0px')]}
]))

,MSE,R2
Test Year,,
2021,"6,372,397,896.7095",0.7591
2022,"17,054,725,204.2593",0.4117
2023,"25,678,395,498.5468",0.1290


Yes, model degradation occured for the deep learning model, it can be seen from both the test MSE and R2 error.

The MSE increased from 6,372,397,896.7095 to 17,054,725,054.4132 to 25,678,395,375.0838

The R2 dropped from 0.7591 to 0.4117 to 0.1290



---



---



4.Model degradation could be caused by [various data distribution shifts](https://huyenchip.com/2022/02/07/data-distribution-shifts-and-monitoring.html#data-shift-types): covariate shift (features), label shift and/or concept drift (altered relationship between features and labels).
There are various conflicting terminologies in the [literature](https://www.sciencedirect.com/science/article/pii/S0950705122002854#tbl1). Let’s stick to this reference for this assignment.

> Using the **Alibi Detect** library, apply the **TabularDrift** function with the training data (year 2019 and before) used as the reference and **detect which features have drifted** in the 2023 test dataset. Before running the statistical tests, ensure you **sample 800 data points** each from the train and test data. Do not use the whole train/test data. (Hint: use this example as a guide https://docs.seldon.io/projects/alibi-detect/en/stable/examples/cd_chi2ks_adult.html)


In [12]:
# YOUR CODE HERE
train_sample = train_df.drop(columns = prediction_target).sample(n = 800, random_state = SEED).copy()
test_sample = test_df_2023.drop(columns = prediction_target).sample(n = 800, random_state = SEED).copy()

encoder = OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1)
train_sample[categorical_features] = encoder.fit_transform(train_sample[categorical_features])
test_sample[categorical_features]  = encoder.transform(test_sample[categorical_features])

In [13]:
categories_per_feature = {
    i + len(numeric_features): None
    for i in range(len(categorical_features))
}

cd = TabularDrift(
    train_sample.values,
    p_val = 0.05,
    categories_per_feature = categories_per_feature
)

result = cd.predict(test_sample.values, drift_type = 'feature')

In [14]:
all_detect_features = numeric_features + categorical_features
drift_results = pd.DataFrame({
    'Feature':  all_detect_features,
    'Test':     ['K-S' if i < len(numeric_features) else 'Chi2' for i in range(len(all_detect_features))],
    'p-value':  result['data']['p_val'].round(4),
    'Drifted?': ['YES' if d else 'NO' for d in result['data']['is_drift']]
})

drift_results.index = range(1, len(drift_results) + 1)
drift_results.index.name = 'No.'

display(drift_results.style.apply(
    lambda x: ['background-color: #ffcccc' if v == 'YES' else '' for v in x],
    subset = ['Drifted?']
))

print(f"\nTotal features drifted: {sum(result['data']['is_drift'])} / {len(all_detect_features)}")

,Feature,Test,p-value,Drifted?
No.,,,,
1,dist_to_nearest_stn,K-S,0.416300,NO
2,dist_to_dhoby,K-S,0.015500,YES
3,degree_centrality,K-S,0.655600,NO
4,eigenvector_centrality,K-S,0.028400,YES
5,remaining_lease_years,K-S,0.000000,YES
6,floor_area_sqm,K-S,0.107800,NO
7,month,Chi2,0.000000,YES
8,town,Chi2,0.025700,YES
9,flat_model_type,Chi2,0.004400,YES



Total features drifted: 6 / 10


5. Assuming that the flurry of housing measures have made an impact on the relationship between all the features and resale_price (i.e. P(Y|X) changes), which type of data distribution shift possibly led to model degradation?


In [15]:
# YOUR ANSWER HERE

Concept Drift possibly led to model degradation. When P(Y|X) changes, it means that the relationship between the features and the resale price has changed.

Although the features (X) may remain the same distribution, the model's learned mapping is no longer valid because the underlying relationship has shifted due to external factors, for example housing policy.

In this context, even if a flat has the same feature as a flat in the training data (2019 and before), such as floor_area_sqm or dist_to_nearest_stn..., the resale price in 2023 would still be different due to the fact that new concepts have fundamentally changed how buyers value these features. For example, stricter loan limits may reduce the weight buyers place on floor_area_sqm.

The model trained on data before 2020 cannot capture these new relationships, leading to model degradation when it is evaluated on data from year 2023.

6. From your analysis via TabularDrift, which features contribute to this shift?


In [16]:
# YOUR ANSWER HERE

Based on the drift detection table, 6 out of 10 features have drifted significantly, with p-value < 0.05.


Continuous features:

1. dist_to_dhoby (p = 0.0155)
2. eigenvector_centrality (p = 0.0284)
3. remaining_lease_years (p = 0.0000)

Categorical features (Chi2 test):

1. town (p = 0.0257)
2. flat_model_type (p = 0.0044)

Even though month (p = 0.0000) also had p-value of lower than 0.05, it is not a meaningful find as month is a cyclical feature that repeats every year. The drift detected in month is likely due to sampling bias, it's very likely that the 800 samples drawn from the training data may have a different monthly distribution than the 800 samples drawn from 2023.

7. Suggest 1 way to address model degradation and implement it, showing improved test R2 for year 2023.


1 way to address this issue is to train the data with more recent data, in our case, we will be using the data from the year 2021 and before as training data and data from 2022 as validation set.

In [17]:
# YOUR CODE HERE
new_train_df = df[df['year'] <= 2021][all_features]
new_val_df = df[df['year'] == 2022][all_features]
batch_size = 1024
max_epochs = 50
layers = "50"

Due to the same reason as auto_lr_find being turned off in Part B1, we will be turning it off and using a learning rate of 0.01 here:

In [18]:
data_config = DataConfig(
    target = prediction_target,
    continuous_cols = numeric_features,
    categorical_cols = categorical_features
)

trainer_config = TrainerConfig(
    auto_lr_find = False,
    batch_size = batch_size,
    max_epochs = max_epochs
)

model_config = CategoryEmbeddingModelConfig(
    task = "regression",
    layers = layers
)

optimizer_config = OptimizerConfig()

new_tabular_model = TabularModel(
    data_config = data_config,
    model_config = model_config,
    optimizer_config = optimizer_config,
    trainer_config = trainer_config
)

new_tabular_model.config['learning_rate'] = 0.01

2026-03-14 13:17:42,033 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [19]:
new_tabular_model.fit(train = new_train_df, validation = new_val_df)

INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-03-14 13:17:44,609 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-14 13:17:44,684 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-14 13:17:44,908 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-14 13:17:44,975 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-14 13:17:45,021 - {pytorch_tabular.tabular_model:677} - INFO - Training Started

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  3.0 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.6 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.6 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.6 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is 
set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

2026-03-14 13:21:20,169 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-14 13:21:20,170 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


In [20]:
new_evaluation = new_tabular_model.evaluate(test_df_2023)
new_pred_df = new_tabular_model.predict(test_df_2023)

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │       20450668544.0       │
│  test_mean_squared_error  │       20450668544.0       │
└───────────────────────────┴───────────────────────────┘

In [21]:
new_test_values  = test_df_2023["resale_price"].values
new_pred_values  = new_pred_df["resale_price_prediction"].values

# Calculate MSE and R2
new_test_mse = mean_squared_error(new_test_values, new_pred_values)
new_test_r2  = r2_score(new_test_values, new_pred_values)

print(f"Test MSE: {new_test_mse:.4f}")
print(f"Test R2:  {new_test_r2:.4f}")

Test MSE: 20450668334.6788
Test R2:  0.3063


### Appendix A



Here are our results from a linear regression model. We used StandardScaler for continuous variables and OneHotEncoder for categorical variables.

While 2021 data can be predicted well, test R2 dropped rapidly for 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| Year <= 2020 | 2021     | 0.76    |
| Year <= 2020 | **2022**     | 0.41    |
| Year <= 2020 | **2023**     | **0.10**   |



Similarly, a model trained on 2017 data can predict 2018-2021 well (with slight degradation in performance for 2021), but drops drastically in 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2017         | 2018     | 0.90    |
|              | 2019     | 0.89    |
|              | 2020     | 0.87    |
|              | 2021     | 0.72    |
|              | **2022**     | **0.37**    |
|              | **2023**     | **0.09**    |

With the test set fixed at year 2021, training on data from 2017-2020 still works well on the test data, with minimal degradation. Training sets closer to year 2021 generally do better.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2020         | 2021     | 0.81    |
| 2019         | 2021     | 0.75    |
| 2018         | 2021     | 0.73    |
| 2017         | 2021     | 0.72    |